<a href="https://colab.research.google.com/github/berenisbecker-a11y/movies_/blob/main/query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [7]:
from google.colab import files
uploaded = files.upload()

Saving keywords.csv to keywords.csv
Saving links_small.csv to links_small.csv
Saving movies_metadata.csv to movies_metadata (1).csv
Saving ratings_small.csv to ratings_small.csv


In [2]:
import os
os.listdir()

['.config', 'sample_data']

In [9]:
import pandas as pd

movies = pd.read_csv("movies_metadata.csv")
ratings = pd.read_csv("ratings_small.csv")
keywords = pd.read_csv("keywords.csv")
links = pd.read_csv("links_small.csv")

/tmp/ipykernel_17237/1596502740.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv("movies_metadata.csv")


In [10]:
print(movies.columns)
print(ratings.columns)
print(keywords.columns)
print(links.columns)

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')
Index(['id', 'keywords'], dtype='object')
Index(['movieId', 'imdbId', 'tmdbId'], dtype='object')


In [11]:
import pandas as pd

# Cambiar columnas a formato a número y eliminar errores.
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
ratings['movieId']= pd.to_numeric(ratings['movieId'], errors='coerce')
keywords['id']= pd.to_numeric(keywords['id'], errors='coerce')

In [12]:
movies= movies.dropna(subset=['id'])
links= links.dropna(subset=['tmdbId'])
ratings= ratings.dropna(subset=['movieId'])
keywords= keywords.dropna(subset=['id'])

In [13]:
movies['id']= movies['id'].astype(int)
links['tmdbId']= links['tmdbId'].astype(int)
ratings['movieId']= ratings['movieId'].astype(int)
keywords['id']= keywords['id'].astype(int)

In [54]:
# Unir tablas a partir de columnas relacionadas.
movies_links = pd.merge(movies, links, left_on='id', right_on='tmdbId')
movies_links_ratings = pd.merge(movies_links, ratings, left_on='movieId', right_on='movieId')
full_data = pd.merge(movies_links_ratings, keywords, left_on='id', right_on= 'id')
full_data.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,video,vote_average,vote_count,movieId,imdbId,tmdbId,userId,rating,timestamp,keywords
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,False,7.7,5415.0,1,114709,862,7,3.0,851866703,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,False,7.7,5415.0,1,114709,862,9,4.0,938629179,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
2,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,False,7.7,5415.0,1,114709,862,13,5.0,1331380058,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
3,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,False,7.7,5415.0,1,114709,862,15,2.0,997938310,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
4,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,False,7.7,5415.0,1,114709,862,19,3.0,855190091,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."


In [15]:
rating_por_pelicula = full_data.groupby('id').agg({'rating': 'mean', 'userId': 'count'}).reset_index()
rating_por_pelicula = rating_por_pelicula.sort_values(by='rating', ascending=False)
rating_por_pelicula.to_csv('rating_por_pelicula.csv', index=False)
rating_por_pelicula = rating_por_pelicula.rename(columns={
    'rating': 'rating_promedio',
    'userId': 'cantidad_ratings'
})

rating_por_pelicula.head()

,id,rating_promedio,cantidad_ratings
8371,128158,5.0,1
4778,17473,5.0,1
4709,16876,5.0,1
4746,17200,5.0,1
4761,17343,5.0,1


In [16]:
titulos_peliculas = full_data[['id', 'original_title']].drop_duplicates()

rating_por_pelicula_dashboard = pd.merge(
    rating_por_pelicula,
    titulos_peliculas,
    on='id',
    how='left'
)

rating_por_pelicula_dashboard = rating_por_pelicula_dashboard.rename(columns={
    'rating': 'rating_promedio',
    'userId': 'cantidad_ratings'
})

rating_por_pelicula_dashboard = rating_por_pelicula_dashboard[
    ['id', 'original_title', 'rating_promedio', 'cantidad_ratings']
]

rating_por_pelicula_dashboard.to_csv('/content/rating_por_pelicula_dashboard.csv', index=False)

rating_por_pelicula_dashboard.head()

,id,original_title,rating_promedio,cantidad_ratings
0,128158,"Syngué sabour, pierre de patience",5.0,1
1,17473,The Room,5.0,1
2,16876,Bang Bang You're Dead,5.0,1
3,17200,Helter Skelter,5.0,1
4,17343,Vampyros Lesbos,5.0,1


In [17]:
# Importar JSON para extraer el género de una cadena de texto.
import pandas as pd
import json
import ast

def parse_genres(cell):
    if pd.isna(cell):
        return None

    if isinstance(cell, list):
        return cell

    if not isinstance(cell, str):
        return None

    if cell.strip() == "":
        return None

    try:
        return json.loads(cell)
    except (json.JSONDecodeError, TypeError):
        try:
            return ast.literal_eval(cell)
        except (ValueError, SyntaxError, TypeError):
            return None

full_data['genres_parseados'] = full_data['genres'].apply(parse_genres)

full_data = full_data.dropna(subset=['genres_parseados']).copy()

full_data = full_data[
    full_data['genres_parseados'].apply(lambda x: isinstance(x, list) and len(x) > 0)
].copy()

def extraer_generos(lista_generos):
    generos = []

    for genero in lista_generos:
        if isinstance(genero, dict) and 'name' in genero:
            generos.append(genero['name'])

    return generos

full_data['genre_names'] = full_data['genres_parseados'].apply(extraer_generos)

full_data = full_data[
    full_data['genre_names'].apply(lambda x: isinstance(x, list) and len(x) > 0)
].copy()

full_data_generos = full_data.explode('genre_names').copy()

full_data_generos = full_data_generos.dropna(subset=['genre_names']).copy()

full_data_generos = full_data_generos.rename(columns={
    'genre_names': 'genre'
})

full_data_generos[['title', 'genre']].head(20)

,title,genre
0,Toy Story,Animation
0,Toy Story,Comedy
0,Toy Story,Family
1,Toy Story,Animation
1,Toy Story,Comedy
1,Toy Story,Family
2,Toy Story,Animation
2,Toy Story,Comedy
2,Toy Story,Family
3,Toy Story,Animation


In [18]:
rating_por_genero = full_data_generos.groupby('genre').agg(
    rating_promedio=('rating', 'mean'),
    cantidad_ratings=('rating', 'count')
).reset_index()

rating_por_genero = rating_por_genero.sort_values(by='rating_promedio',ascending=False)

rating_por_genero

,genre,rating_promedio,cantidad_ratings
10,History,3.852897,3970
5,Documentary,3.832074,1456
9,Foreign,3.798544,206
18,War,3.789849,3783
6,Drama,3.672849,47151
2,Animation,3.629009,6143
4,Crime,3.624708,17541
13,Mystery,3.586836,8804
12,Music,3.571064,3919
19,Western,3.562733,1610


In [37]:
full_data_generos['release_date'] = pd.to_datetime(
    full_data_generos['release_date'],
    errors='coerce')

full_data_generos['year'] = full_data_generos['release_date'].dt.year

tendencia_genero_anio = full_data_generos.groupby(['year', 'genre']).agg(
    rating_promedio=('rating', 'mean'),
    cantidad_ratings=('rating', 'count')
).reset_index()

# Filtrar grupos con pocos ratings para que no distorsionen.
tendencia_genero_anio = tendencia_genero_anio[
    tendencia_genero_anio['cantidad_ratings'] >= 10

]

tendencia_genero_anio.head(20)

,year,genre,rating_promedio,cantidad_ratings
17,1920,Crime,4.000000,13
18,1920,Drama,3.928571,14
19,1920,Horror,3.750000,14
22,1920,Thriller,3.750000,14
24,1921,Comedy,4.416667,12
32,1922,Fantasy,3.809524,21
33,1922,Horror,3.840909,22
49,1925,Adventure,4.052632,19
50,1925,Comedy,4.033333,15
51,1925,Drama,3.916667,36


In [38]:
full_data['revenue'] = pd.to_numeric(full_data['revenue'], errors='coerce')
full_data['budget'] = pd.to_numeric(full_data['budget'], errors='coerce')

In [39]:
# Calcular diferencia de revenue y budget para ver la rentabilidad.

# Agrupar por película.
revenue_data = full_data.groupby(['id', 'original_title']).agg(revenue_promedio=('revenue', 'mean'),budget_promedio=('budget', 'mean')
).reset_index()

# Crear la diferencia después del groupby.
revenue_data['diferencia_revenue'] = (revenue_data['revenue_promedio'] - revenue_data['budget_promedio'])

# Redondear columnas.
revenue_data['revenue_promedio'] = revenue_data['revenue_promedio'].round(2)
revenue_data['budget_promedio'] = revenue_data['budget_promedio'].round(2)
revenue_data['diferencia_revenue'] = revenue_data['diferencia_revenue'].round(2)

revenue_data = revenue_data.sort_values(
    by='diferencia_revenue',
    ascending=False)

revenue_data.head(20)

,id,original_title,revenue_promedio,budget_promedio,diferencia_revenue
5125,19995,Avatar,2.787965e+09,237000000.0,2.550965e+09
8411,140607,Star Wars: The Force Awakens,2.068224e+09,245000000.0,1.823224e+09
398,597,Titanic,1.845034e+09,200000000.0,1.645034e+09
8384,135397,Jurassic World,1.513529e+09,150000000.0,1.363529e+09
8500,168259,Furious 7,1.506249e+09,190000000.0,1.316249e+09
5568,24428,The Avengers,1.519558e+09,220000000.0,1.299558e+09
3807,12445,Harry Potter and the Deathly Hallows: Part 2,1.342000e+09,125000000.0,1.217000e+09
8206,99861,Avengers: Age of Ultron,1.405404e+09,280000000.0,1.125404e+09
8252,109445,Frozen,1.274219e+09,150000000.0,1.124219e+09
8626,211672,Minions,1.156731e+09,74000000.0,1.082731e+09


In [66]:
# Ver la diferencia de revenue y budget en millones.
revenue_data['diferencia_millones'] = (
    revenue_data['diferencia_revenue'] / 1000000
).round(2)

revenue_data[['original_title', 'revenue_promedio', 'budget_promedio', 'diferencia_millones']].head()

,original_title,revenue_promedio,budget_promedio,diferencia_millones
5125,Avatar,2.787965e+09,237000000.0,2550.97
8411,Star Wars: The Force Awakens,2.068224e+09,245000000.0,1823.22
398,Titanic,1.845034e+09,200000000.0,1645.03
8384,Jurassic World,1.513529e+09,150000000.0,1363.53
8500,Furious 7,1.506249e+09,190000000.0,1316.25


In [79]:
import pandas as pd

# Calcular popularidad por compañía de producción utilizando el comando JSON.

# Select relevant columns from full_data for this analysis
production_company_data = full_data[['id', 'original_title', 'production_companies', 'popularity']].copy()

production_company_data['company_names'] = production_company_data['production_companies'].apply(parse_genres)
production_company_data['company_names'] = production_company_data['company_names'].apply(extraer_generos)

peliculas_companies = production_company_data.explode('company_names').copy()

peliculas_companies = peliculas_companies.dropna(subset=['company_names']).copy()

peliculas_companies = peliculas_companies.rename(columns={
    'company_names': 'production_company'
})

peliculas_companies[['original_title', 'production_company', 'popularity']].head(20)

,original_title,production_company,popularity
0,Toy Story,Pixar Animation Studios,21.946943
1,Toy Story,Pixar Animation Studios,21.946943
2,Toy Story,Pixar Animation Studios,21.946943
3,Toy Story,Pixar Animation Studios,21.946943
4,Toy Story,Pixar Animation Studios,21.946943
5,Toy Story,Pixar Animation Studios,21.946943
6,Toy Story,Pixar Animation Studios,21.946943
7,Toy Story,Pixar Animation Studios,21.946943
8,Toy Story,Pixar Animation Studios,21.946943
9,Toy Story,Pixar Animation Studios,21.946943


In [80]:
top_productoras = peliculas_companies.groupby('production_company').agg(
    cantidad_peliculas=('id', 'count')
).reset_index()

top_productoras = top_productoras.sort_values(
    by='cantidad_peliculas',
    ascending=False
)

top_productoras.head(20)

,production_company,cantidad_peliculas
7069,Warner Bros.,12059
4929,Paramount Pictures,9985
6851,Universal Pictures,8906
6748,Twentieth Century Fox Film Corporation,8660
1446,Columbia Pictures,4205
4560,New Line Cinema,3726
4321,Miramax Films,3352
6658,Touchstone Pictures,3325
7052,Walt Disney Pictures,3125
1447,Columbia Pictures Corporation,3029


In [81]:
popularidad_por_productora = peliculas_companies.groupby('production_company').agg(
    popularidad_promedio=('popularity', 'mean'),
    cantidad_peliculas=('id', 'count')
).reset_index()

popularidad_por_productora = popularidad_por_productora[
    popularidad_por_productora['cantidad_peliculas'] >= 10
]

popularidad_por_productora['popularidad_promedio'] = popularidad_por_productora['popularidad_promedio'].round(2)

popularidad_por_productora = popularidad_por_productora.sort_values(
    by='popularidad_promedio',
    ascending=False
)

popularidad_por_productora.head(10)

,production_company,popularidad_promedio,cantidad_peliculas
6464,The Donners' Company,187.86,18
3962,MJW Films,183.87,10
1760,DefyNite Films,183.87,10
68,87Eleven,183.87,10
3511,Kinberg Genre,165.13,21
441,Artemple - Hollywood,154.80,25
4857,Pacific Standard,134.85,29
3116,Illumination Entertainment,98.20,34
5821,Shaw Brothers,92.03,153
71,A Band Apart,78.37,672


In [82]:
full_data['rating'] = pd.to_numeric(full_data['rating'], errors='coerce')
full_data['popularity'] = pd.to_numeric(full_data['popularity'], errors='coerce')

popularidad_vs_rating = full_data.groupby(['id', 'original_title']).agg(
    rating_promedio=('rating', 'mean'),
    popularity=('popularity', 'mean'),
    cantidad_ratings=('rating', 'count')
).reset_index()

popularidad_vs_rating['rating_promedio'] = popularidad_vs_rating['rating_promedio'].round(2)
popularidad_vs_rating['popularity'] = popularidad_vs_rating['popularity'].round(2)

popularidad_vs_rating = popularidad_vs_rating.sort_values(
    by='popularity',
    ascending=False
)

popularidad_vs_rating.head(20)

,id,original_title,rating_promedio,popularity,cantidad_ratings
8654,211672,Minions,2.90,547.49,5
8554,177572,Big Hero 6,4.10,213.85,21
8895,293660,Deadpool,3.39,187.86,18
5126,19995,Avatar,3.64,185.07,67
8747,245891,John Wick,3.45,183.87,10
8650,210577,Gone Girl,3.72,154.80,25
8384,131631,The Hunger Games: Mockingjay - Part 1,2.90,147.10,10
8843,271110,Captain America: Civil War,3.57,145.88,7
476,680,Pulp Fiction,4.26,140.95,324
102,155,The Dark Knight,4.24,123.17,121


In [83]:
# Filtrar películas con mínimo de ratings para que no se distorsionen.
popularidad_vs_rating_filtrada = popularidad_vs_rating[
    popularidad_vs_rating['cantidad_ratings'] >= 50
]

popularidad_vs_rating_filtrada.head(20)

,id,original_title,rating_promedio,popularity,cantidad_ratings
5126,19995,Avatar,3.64,185.07,67
476,680,Pulp Fiction,4.26,140.95,324
102,155,The Dark Knight,4.24,123.17,121
39,78,Blade Runner,4.04,96.27,146
360,550,Fight Club,4.18,63.87,202
200,278,The Shawshank Redemption,4.49,51.65,311
5,13,Forrest Gump,4.05,48.31,341
13,22,Pirates of the Caribbean: The Curse of the Bla...,3.85,47.33,141
3,11,Star Wars,4.22,42.15,291
288,424,Schindler's List,4.30,41.73,244


In [84]:
popularidad_vs_rating_tableau = popularidad_vs_rating_filtrada[
    ['id', 'original_title', 'rating_promedio', 'popularity', 'cantidad_ratings']
].drop_duplicates()

popularidad_vs_rating_tableau.to_csv(
    '/content/popularidad_vs_rating_tableau.csv',
    index=False
)

In [85]:
from google.colab import files

files.download('/content/popularidad_vs_rating_tableau.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>